# Verificación — Spark lee tu datalake

**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Estudiante:** Sebastián Durán Fernández — sduranf@eafit.edu.co
**Fecha:** 2026-08-13 · **Clúster:** `j-1ZDUFFV7I93JN` · **App:** `application_1786654608050_0001`

## Objetivo

Cerrar el Lab 1a confirmando que tu clúster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.

**Qué debe verse al final para confirmar que el lab está completo:**
- La Celda 2 muestra el schema y 5 filas del Parquet leído desde S3
  (si esto funciona, tu bucket, tu rol IAM y tu clúster están bien
  configurados de punta a punta).
- La Celda 3 imprime el tiempo de una misma consulta en Parquet y en
  CSV, y el ratio entre ambos.
- Completaste el análisis de la Celda 4 y capturaste el DAG de Spark UI
  como indica la Celda 5.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en EMR (donde ya existe una sesión activa
# administrada por el clúster) como en un entorno local con pyspark
# instalado, sin necesitar ramas de código distintas.
spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

# EDITAR: reemplaza por el bucket que creaste en setup_s3.sh
# (convención: st1630-{tu-usuario}-{año})
BUCKET = "st1630-sduranf-2026"

# En EMR, S3 se referencia directamente con el esquema s3://.
# En local (con las credenciales de AWS Academy exportadas), la misma
# ruta también funciona porque Spark usa el conector S3A por debajo.
ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

# Si ves el schema y las filas de arriba, tu datalake funciona
# correctamente de punta a punta: bucket, permisos IAM y clúster EMR.
print("Filas leídas:", df_parquet.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 20:59:21 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


root
 |-- order_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unit: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- devuelto: boolean (nullable = true)



+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|order_id  |fecha     |region      |producto |categoria  |cantidad|precio_unit|total    |canal |devuelto|
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|ORD-000001|2026-04-22|Cali        |Mouse    |Electrónica|1       |789300.0   |789300.0 |online|false   |
|ORD-000002|2026-03-22|Barranquilla|Gorra    |Ropa       |2       |57000.0    |114000.0 |tienda|false   |
|ORD-000003|2026-01-06|Bogotá      |Zapatos  |Ropa       |4       |163800.0   |655200.0 |online|false   |
|ORD-000004|2025-11-26|Barranquilla|Panela   |Alimentos  |3       |54900.0    |164700.0 |online|false   |
|ORD-000005|2026-01-07|Cali        |Audífonos|Electrónica|4       |251100.0   |1004400.0|tienda|true    |
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
only showing top 5 rows

Filas leídas: 10000


In [2]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 1.925 s (6 filas de resultado)


CSV: 1.062 s (6 filas de resultado)

Ratio CSV / Parquet: 0.55x


## Análisis — completa antes de entregar

### a) Tamaño en disco

¿Cuánto pesa `prueba_parquet.parquet` frente a `prueba_csv.csv`?

→ Parquet pesa **185,0 KiB** y CSV **798,3 KiB** para las mismas 10.000 filas:
el CSV ocupa **4,32 veces más**. La diferencia viene de que Parquet almacena por
columnas y comprime cada una con el códec adecuado a su tipo (snappy sobre
valores homogéneos), mientras que el CSV guarda todo como texto y repite los
valores de baja cardinalidad —`region`, `categoria`, `canal`— en cada fila.

### b) Tiempo de la consulta

¿Cuánto tardó la consulta de la Celda 3 en cada formato?

→ **Parquet: 1,925 s** · **CSV: 1,062 s**. La consulta fue idéntica en ambos
casos (filtro por `region` y `categoria`, `groupBy` sobre `producto`, suma de
`total` y ordenamiento), y devolvió 6 filas de resultado en los dos formatos.

### c) Ratio de mejora

¿Cuál fue el ratio de mejora (CSV / Parquet)? ¿Coincide con el orden de
magnitud visto en clase (~9x)?

→ El ratio fue **0,55x**, es decir, el resultado se invirtió: Parquet tardó casi
el doble que CSV, lejos del ~9x a favor de Parquet visto en clase. Revisé si era
un error de ejecución y concluí que el resultado es correcto; lo que falla es la
expectativa, por dos motivos que se suman.

El primero es el **tamaño del dataset**: 185 KB no llenan ni un bloque de
lectura, así que el tiempo no lo consume el I/O sino la planificación y el
arranque de tareas. El DAG lo muestra con claridad: de los 1,9 s totales, el
`Scan parquet` solo tardó **79 ms**, y el `Exchange` movió **457 bytes**
repartidos en **1.000 particiones** configuradas por defecto para producir 6
filas. Con esa proporción, cualquier ventaja de compresión y *column pruning*
queda enterrada bajo el overhead fijo del motor.

El segundo es el **orden del experimento**: Parquet se midió primero y pagó el
arranque de los executors y el calentamiento de la JVM, mientras que el CSV corrió
con el clúster ya caliente. Además, el `inferSchema` del CSV se ejecutó al crear
el DataFrame —visible como la query 2 del Spark UI, 0,6 s— es decir, **fuera del
cronómetro**, de modo que el costo de deducir tipos no se le imputó al CSV.

La conclusión que saco es que la ventaja de Parquet no es una propiedad
incondicional del formato, sino que aparece cuando el I/O domina el tiempo total:
con datasets de GB, muchas columnas y consultas selectivas, el *predicate
pushdown* y el *column pruning* sí deciden el resultado. A esta escala, el
experimento mide el overhead de Spark, no el formato.

### d) Conexión con el Teorema CAP

¿Por qué S3 con replicación entre múltiples zonas de disponibilidad es
una decisión **CP**? ¿Qué sacrifica a cambio?

→ Porque S3 no confirma una escritura hasta que el objeto está durablemente
replicado en varias zonas de disponibilidad, y desde 2020 garantiza consistencia
*read-after-write*: una lectura posterior a un `PUT` exitoso siempre devuelve la
versión más reciente. Ante una partición de red entre zonas, S3 prefiere no
completar la operación antes que confirmar una escritura que podría no estar
replicada o servir una versión desactualizada del objeto.

Lo que sacrifica es **disponibilidad de escritura y latencia**: cada `PUT` espera
la confirmación de varias zonas, de modo que responde más lento y, ante fallos de
red, puede rechazar operaciones que un sistema AP —como DynamoDB en modo de
consistencia eventual— sí aceptaría, a cambio de que algunos lectores vieran datos
obsoletos durante un rato. Para un datalake es el intercambio correcto: si Bronze
pudiera devolver una versión vieja de un archivo, todas las capas derivadas
—Silver y Gold— quedarían construidas sobre datos que nadie puede reproducir.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

## Bitácora de delegación

| Tarea | ¿Delegado a agente? | Herramienta | Justificación |
|---|---|---|---|
| Boilerplate de SparkSession / lectura de S3 | No | — | El código venía provisto en el notebook base del curso; solo edité la variable `BUCKET` con el nombre de mi bucket. |
| Diseño de la consulta del benchmark (Celda 3) | No | — | La consulta también venía provista en el notebook base, sin modificaciones. |
| Ejecución del notebook contra el clúster EMR | Sí | Claude Code (Opus 5) | Operación mecánica: subir el notebook al master por SSH y ejecutarlo con `nbconvert`. Los resultados son de mi propio clúster (`application_1786654608050_0001`). |
| Interpretación de los resultados (Celda 4) | Parcial — redacción asistida | Claude Code (Opus 5) | Las conclusiones son mías: que el resultado invertido se explica por el tamaño del dataset y el orden del experimento, y la lectura de CAP. El agente aportó los datos medidos del DAG y convirtió mis respuestas en prosa. |
| Troubleshooting de errores de conexión a S3 | Sí | Claude Code (Opus 5) | No hubo errores de S3, pero sí de entorno (rutas de Git Bash en `file://`, incompatibilidad `--use-default-roles` con `InstanceProfile`, apertura del puerto 22 del security group). El lab clasifica este troubleshooting como delegable. |

> Nota: el clúster se apagó (`terminate-clusters`) inmediatamente después de
> capturar el DAG. Tiempo total encendido: ~45 minutos.